# Simple EDA — operating-room data

This notebook explores `donees bloc anonyme pour centrale 2026.xlsx`. It covers data quality, activity over time, patient age, length of stay, intervention/anesthesia categories, and operating-room timing.

> Privacy: patient, case, and practitioner identifiers are excluded from row-level previews and charts.

In [ ]:
# If needed, run once:
# %pip install pandas openpyxl matplotlib seaborn

from pathlib import Path
from datetime import datetime, time
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
DATA_FILE = Path("resources/donees bloc anonyme pour centrale 2026.xlsx")
assert DATA_FILE.exists(), f"File not found: {DATA_FILE.resolve()}"

## 1. Load and prepare the data

In [ ]:
df_raw = pd.read_excel(DATA_FILE, engine="openpyxl")
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
display(pd.DataFrame({"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}))

In [ ]:
def clean_name(name):
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

df = df_raw.copy()
df.columns = [clean_name(c) for c in df.columns]

date_cols = ["date_entree", "date_sortie", "date_naissance", "date_inter"]
for col in date_cols:
    if col in df:
        df[col] = pd.to_datetime(df[col], errors="coerce")

safe_preview = df.drop(columns=["no_cas", "id_patient", "praticien", "nom_chir"], errors="ignore")
display(safe_preview.head())

## 2. Data quality

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(1),
    "unique_n": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(quality)

In [ ]:
missing = quality.query("missing_pct > 0").sort_values("missing_pct")
if not missing.empty:
    ax = missing["missing_pct"].plot.barh(figsize=(9, max(4, len(missing) * 0.28)), color="#4C78A8")
    ax.set(title="Missing values by column", xlabel="Missing (%)", ylabel="")
    plt.tight_layout()
    plt.show()

### Missing-value recoverability

A missing value can often be represented safely in a model, but this does **not** mean that its original clinical value can be recovered. The table below separates three situations:

- **Structural or undocumented:** an empty secondary diagnosis or procedure position can be retained as `no additional code documented`; a specific code must not be invented.
- **Ambiguous:** absence and non-documentation cannot be distinguished from this workbook alone.
- **Not recoverable:** the exact value cannot be inferred reliably; use an explicit unknown category, a missingness indicator, or exclude the row only from analyses that require that field.

Mode imputation is inappropriate for all clinical codes and free-text intervention labels because it would fabricate clinical information and distort category frequencies.

In [ ]:
secondary_diagnoses = [f"cim_assoc_{i}" for i in range(1, 6)]
secondary_procedures = [f"ccam_{i}" for i in range(2, 5)]

recoverability = {
    **{col: "Structural or undocumented" for col in secondary_diagnoses},
    **{col: "Structural or undocumented" for col in secondary_procedures},
    "anesth_loco_reg": "Ambiguous",
    "ccam_1": "Not recoverable",
    "praticien": "Not recoverable",
    "nom_chir": "Not recoverable",
    "interv_type": "Not recoverable reliably",
    "anesth_type": "Not recoverable",
}

recommended_handling = {
    **{col: "Keep as no additional diagnosis documented; never impute a code" for col in secondary_diagnoses},
    **{col: "Keep as no additional procedure documented; never impute a code" for col in secondary_procedures},
    "anesth_loco_reg": "Use none/not documented plus a missingness flag; confirm the local convention",
    "ccam_1": "Use unknown or exclude only where a principal procedure code is essential",
    "praticien": "Use unknown; exclude only practitioner-level analyses",
    "nom_chir": "Use unknown; exclude only surgeon-level analyses",
    "interv_type": "Use CCAM/GHM/CIM fields where available, otherwise unknown; do not label-impute",
    "anesth_type": "Use unknown plus a missingness flag; do not infer it from locoregional anesthesia",
}

missing_assessment = quality.loc[quality["missing_n"].gt(0), ["missing_n", "missing_pct"]].copy()
missing_assessment["recoverability"] = missing_assessment.index.map(recoverability)
missing_assessment["recommended_handling"] = missing_assessment.index.map(recommended_handling)
display(missing_assessment.sort_values(["recoverability", "missing_pct"]))

#### Why missing `Interv Type` values should not be filled automatically

`Interv Type` is a high-cardinality free-text field rather than a standardized code. The audit below tests whether its missing values could at least be recovered deterministically from the principal CCAM code. A CCAM code associated with several intervention labels is considered ambiguous. Even a one-label match is only a candidate requiring domain validation, not proof of the missing label.

In [ ]:
missing_intervention = df[df["interv_type"].isna()].copy()
labeled_ccam = df[df["interv_type"].notna() & df["ccam_1"].notna()]
ccam_label_counts = labeled_ccam.groupby("ccam_1")["interv_type"].agg(
    observed_labels="nunique",
    reference_rows="size",
)
intervention_audit = missing_intervention[["ccam_1"]].join(ccam_label_counts, on="ccam_1")

audit_summary = pd.Series({
    "Missing Interv Type": len(missing_intervention),
    "Also missing CCAM 1": missing_intervention["ccam_1"].isna().sum(),
    "CCAM 1 present": missing_intervention["ccam_1"].notna().sum(),
    "CCAM 1 maps to one observed label": intervention_audit["observed_labels"].eq(1).sum(),
    "CCAM 1 maps to multiple labels": intervention_audit["observed_labels"].gt(1).sum(),
    "CCAM 1 not seen in labeled rows": (
        intervention_audit["ccam_1"].notna() & intervention_audit["reference_rows"].isna()
    ).sum(),
    "Also missing anesthesia type": missing_intervention["anesth_type"].isna().sum(),
    "Also missing practitioner": missing_intervention["praticien"].isna().sum(),
    "Also missing surgeon": missing_intervention["nom_chir"].isna().sum(),
}, name="rows").to_frame()
display(audit_summary)

### Apply sample exclusions and correct length of stay

The preprocessing requested here removes records with no intervention type or no practitioner. The filters are combined so that records missing both fields are counted only once. The original stay-duration field is preserved, while `duree_sejour_corrigee` is calculated from the admission and discharge dates using the inclusive convention.

In [ ]:
def is_missing_or_blank(series):
    return series.isna() | series.astype("string").str.strip().eq("")

missing_intervention_mask = is_missing_or_blank(df["interv_type"])
missing_practitioner_mask = is_missing_or_blank(df["praticien"])
poor_sample_mask = missing_intervention_mask | missing_practitioner_mask

raw_stay_col = next(c for c in df.columns if c.startswith("duree_sejour"))
recorded_stay = pd.to_numeric(df[raw_stay_col], errors="coerce")
df["duree_sejour_corrigee"] = (df["date_sortie"] - df["date_entree"]).dt.days + 1
df["duree_sejour_incoherente"] = recorded_stay.ne(df["duree_sejour_corrigee"])

filter_summary = pd.Series({
    "Initial rows": len(df),
    "Missing intervention type": missing_intervention_mask.sum(),
    "Missing practitioner": missing_practitioner_mask.sum(),
    "Missing both fields": (missing_intervention_mask & missing_practitioner_mask).sum(),
    "Rows removed (combined rule)": poor_sample_mask.sum(),
    "Rows retained": (~poor_sample_mask).sum(),
    "Removed rows with inconsistent stay duration": (
        poor_sample_mask & df["duree_sejour_incoherente"]
    ).sum(),
}, name="rows").to_frame()

df = df.loc[~poor_sample_mask].copy()

assert df["interv_type"].notna().all()
assert df["praticien"].notna().all()
assert df["duree_sejour_corrigee"].ge(1).all()
display(filter_summary)
print(
    f"Corrected stay durations differing from the source among retained rows: "
    f"{df['duree_sejour_incoherente'].sum():,}"
)

#### Export the cleaned table

The cleaned table is saved separately in the Git-ignored `resources/` directory. The source workbook is never overwritten.

In [ ]:
CLEAN_DATA_FILE = Path("resources/donnees_bloc_nettoyees.xlsx")
df.to_excel(CLEAN_DATA_FILE, index=False, engine="openpyxl")
print(f"Saved {len(df):,} rows and {df.shape[1]} columns to {CLEAN_DATA_FILE}")

## 3. Derived measures

In [ ]:
if {"date_naissance", "date_inter"}.issubset(df.columns):
    df["age_years"] = (df["date_inter"] - df["date_naissance"]).dt.days / 365.25
    df.loc[~df["age_years"].between(0, 110), "age_years"] = np.nan

stay_col = next((c for c in df.columns if c.startswith("duree_sejour") and c != "duree_sejour_corrigee"), None)
if stay_col:
    df["length_of_stay_days_recorded"] = pd.to_numeric(df[stay_col], errors="coerce")
if "duree_sejour_corrigee" in df:
    df["length_of_stay_days"] = df["duree_sejour_corrigee"].astype(float)

def clock_to_minutes(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (datetime, time)):
        return value.hour * 60 + value.minute + value.second / 60
    if isinstance(value, (int, float, np.number)):
        return float(value) * 24 * 60
    parsed = pd.to_datetime(str(value), errors="coerce")
    return np.nan if pd.isna(parsed) else parsed.hour * 60 + parsed.minute + parsed.second / 60

time_candidates = {
    "sspi_pre": next((c for c in df if c.startswith("heure_entree_sspi")), None),
    "room_in": next((c for c in df if c.startswith("heure_d_entree_en_salle")), None),
    "incision": next((c for c in df if c.startswith("heure_incision")), None),
    "room_out": next((c for c in df if c.startswith("heure_de_sortie_de_salle")), None),
}
for short_name, col in time_candidates.items():
    if col:
        df[f"{short_name}_minute"] = df[col].map(clock_to_minutes)

def elapsed_minutes(end, start):
    delta = end - start
    return delta.where(delta >= 0, delta + 24 * 60)

if {"room_in_minute", "room_out_minute"}.issubset(df.columns):
    df["room_duration_min"] = elapsed_minutes(df["room_out_minute"], df["room_in_minute"])
if {"room_in_minute", "incision_minute"}.issubset(df.columns):
    df["entry_to_incision_min"] = elapsed_minutes(df["incision_minute"], df["room_in_minute"])
if {"sspi_pre_minute", "room_in_minute"}.issubset(df.columns):
    df["preop_wait_min"] = elapsed_minutes(df["room_in_minute"], df["sspi_pre_minute"])

for col in ["room_duration_min", "entry_to_incision_min", "preop_wait_min"]:
    if col in df:
        df.loc[~df[col].between(0, 24 * 60), col] = np.nan

### Check for zero-valued operating-room times

The check below uses the raw spreadsheet values, before time conversion. It reports zero values in the four operating-room time fields and their overlap with extreme lengths of stay. Rows for which all four times are zero are then removed from the dataframe used below.

In [ ]:
raw_times = df_raw.copy()
raw_times.columns = [clean_name(c) for c in raw_times.columns]
time_check_cols = {name: col for name, col in time_candidates.items() if col in raw_times}

def is_zero_time(value):
    if pd.isna(value):
        return False
    if isinstance(value, (int, float, np.number)) and value == 0:
        return True
    if isinstance(value, time) and value == time(0, 0):
        return True
    text = str(value).strip().lower()
    return text in {"0", "0.0", "00:00", "00:00:00", "0:00", "0:00:00"}

zero_time_flags = pd.DataFrame(
    {name: raw_times[col].map(is_zero_time) for name, col in time_check_cols.items()},
    index=df.index,
)
any_zero_time_mask = zero_time_flags.any(axis=1)
all_zero_times_mask = zero_time_flags.all(axis=1)
extreme_stay_mask = df["length_of_stay_days"].gt(365)

print(f"Rows with at least one zero-valued time: {any_zero_time_mask.sum():,}")
print(f"Rows with all four times equal to zero: {all_zero_times_mask.sum():,}")
print(f"Rows with length of stay > 365 days: {extreme_stay_mask.sum():,}")
print(
    "Extreme-stay rows with all four times equal to zero: "
    f"{(all_zero_times_mask & extreme_stay_mask).sum():,}"
)
if "interv_type" in df:
    missing_intervention_mask = df["interv_type"].isna()
    print(
        "Missing Interv Type rows with at least one zero-valued time: "
        f"{(missing_intervention_mask & any_zero_time_mask).sum():,}"
    )
    print(
        "Missing Interv Type rows with all four times equal to zero: "
        f"{(missing_intervention_mask & all_zero_times_mask).sum():,}"
    )
display(pd.DataFrame({
    "zero_time_count": zero_time_flags.sum(),
}))

# Exclude records with no usable operating-room time (all four fields are zero).
df = df.loc[~all_zero_times_mask].copy()

## 4. Numerical overview

In [ ]:
measure_cols = [c for c in [
    "age_years", "length_of_stay_days", "room_duration_min",
    "entry_to_incision_min", "preop_wait_min"
] if c in df]
display(df[measure_cols].describe(percentiles=[.25, .5, .75, .9, .95]).T.round(1))

In [ ]:
if measure_cols:
    fig, axes = plt.subplots(len(measure_cols), 1, figsize=(9, 3.2 * len(measure_cols)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, measure_cols):
        upper = df[col].quantile(0.99)
        sns.histplot(df.loc[df[col].between(0, upper), col], bins=35, ax=ax, color="#59A14F")
        ax.set_title(f"Distribution of {col} (up to 99th percentile)")
    plt.tight_layout()
    plt.show()

## 5. Activity over time

In [ ]:
if "date_inter" in df:
    monthly = df.dropna(subset=["date_inter"]).set_index("date_inter").resample("MS").size()
    ax = monthly.plot(figsize=(11, 4), marker="o", markersize=3, color="#F28E2B")
    ax.set(title="Interventions per month", xlabel="Month", ylabel="Number of interventions")
    plt.tight_layout()
    plt.show()

    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekday = df["date_inter"].dt.day_name().value_counts().reindex(weekday_order)
    ax = weekday.plot.bar(figsize=(9, 4), color="#E15759")
    ax.set(title="Interventions by weekday", xlabel="", ylabel="Number of interventions")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Main categorical variables

In [ ]:
categorical_cols = [c for c in [
    "interv_type", "anesth_type", "anesth_loco_reg",
    "sexe", "ghm_code", "cim_diag_pr", "ccam_1"
] if c in df]

for col in categorical_cols:
    counts = df[col].fillna("Missing").astype(str).value_counts().head(15).sort_values()
    ax = counts.plot.barh(figsize=(9, max(3, len(counts) * 0.32)), color="#76B7B2")
    ax.set(title=f"Top categories: {col}", xlabel="Number of records", ylabel="")
    plt.tight_layout()
    plt.show()

## 7. Duration by intervention type

In [ ]:
if {"interv_type", "room_duration_min"}.issubset(df.columns):
    common_types = df["interv_type"].value_counts().head(12).index
    plot_data = df[df["interv_type"].isin(common_types)].copy()
    plot_data = plot_data[plot_data["room_duration_min"] <= plot_data["room_duration_min"].quantile(.99)]
    order = plot_data.groupby("interv_type")["room_duration_min"].median().sort_values().index
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=plot_data, y="interv_type", x="room_duration_min", order=order, showfliers=False)
    plt.title("Operating-room duration for the 12 most frequent intervention types")
    plt.xlabel("Room duration (minutes)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

## 8. Compact summary table

Use this table to identify common procedures with long room occupancy and enough observations for reliable comparison.

In [ ]:
if {"interv_type", "room_duration_min"}.issubset(df.columns):
    summary = (
        df.groupby("interv_type", dropna=False)
          .agg(
              interventions=("interv_type", "size"),
              median_room_min=("room_duration_min", "median"),
              p90_room_min=("room_duration_min", lambda s: s.quantile(.9)),
              median_entry_to_incision_min=("entry_to_incision_min", "median"),
          )
          .sort_values("interventions", ascending=False)
    )
    display(summary.head(20).round(1))

## Notes for interpretation

- Missing timestamps and zero clock values should be checked before operational conclusions are drawn.
- The timing calculations assume a procedure that crosses midnight lasts less than 24 hours.
- Associations in this exploratory notebook are descriptive, not causal.
- Small intervention groups should not be compared without reporting their sample sizes.